In [1]:
!pip install -qU "transformers<4.45.0" "huggingface_hub>=0.23.2,<1.0" FlagEmbedding qdrant-client

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 7.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 62.4 MB/s eta 0:00:00


In [2]:
import os
import gc
import torch
import pandas as pd
from tqdm import tqdm
from qdrant_client import QdrantClient
from qdrant_client.http import models
from FlagEmbedding import BGEM3FlagModel
from huggingface_hub import snapshot_download

2026-03-29 12:22:48.590453: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774786968.770633      25 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774786968.824460      25 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774786969.242573      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774786969.242612      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774786969.242615      25 computation_placer.cc:177] computation placer alr

In [3]:
model = BGEM3FlagModel('/kaggle/input/models/yethukmutt/bge-m3/transformers/m3/1/bge-m3', use_fp16=True) 
corpus_path = "/kaggle/input/datasets/pmtaiii/vietnamese-law-dataset/corpus.csv" 
df_corpus = pd.read_csv(corpus_path)
df_corpus.head()

,text,cid
0,"Thông tư này hướng dẫn tuần tra, canh gác bảo ...",0
1,"1. Hàng năm trước mùa mưa, lũ, Ủy ban nhân dân...",1
2,Tiêu chuẩn của các thành viên thuộc lực lượng ...,2
3,"Nhiệm vụ của lực lượng tuần tra, canh gác đê\n...",3
4,"Phù hiệu của lực lượng tuần tra, canh gác đê\n...",4


In [4]:
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
os.environ['XLA_FLAGS'] = '--xla_gpu_force_compilation_parallelism=1'

In [5]:
client = QdrantClient(path='/kaggle/working/qdrant_db_m3')
COLLECTION_NAME = 'vietnamese_laws_m3'

if client.collection_exists(COLLECTION_NAME): 
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={"dense": models.VectorParams(size=1024, distance=models.Distance.COSINE)},
    sparse_vectors_config={"sparse": models.SparseVectorParams()},
    quantization_config=models.ScalarQuantization(
        scalar=models.ScalarQuantizationConfig(type=models.ScalarType.INT8, always_ram=True)
    )
)

True

In [6]:
BATCH_SIZE = 64
points_batch = []

with tqdm(total=len(df_corpus), desc="Tiến độ") as pbar:
    for i in range(0, len(df_corpus), BATCH_SIZE):
        batch_df = df_corpus.iloc[i: i + BATCH_SIZE]
        texts = batch_df['text'].fillna("").astype(str).tolist()
        cids = batch_df['cid'].tolist()
        embs = model.encode(
            texts, 
            batch_size=BATCH_SIZE,
            return_dense=True,
            return_sparse=True,
            return_colbert_vecs=False,
            max_length=1024
        )

        dense_list = embs['dense_vecs']
        sparse_list = embs['lexical_weights']

        for j in range(len(texts)):
            if not texts[j].strip():
                continue

            sparse_dict = sparse_list[j]
            point = models.PointStruct(
                id=int(cids[j]),
                payload={
                    "content": texts[j],
                    "cid": int(cids[j])
                },
                vector={
                    'dense': dense_list[j].tolist() if hasattr(dense_list[j], 'tolist') else dense_list[j],
                    'sparse': models.SparseVector(
                        indices=[int(k) for k in sparse_dict.keys()],
                        values=list(sparse_dict.values())
                    )
                }
            )
            points_batch.append(point)

        if points_batch:
            client.upsert(collection_name=COLLECTION_NAME, points=points_batch)
            points_batch = []
        
        pbar.update(len(batch_df))
        
        gc.collect()
        if torch.cuda.is_available(): 
            torch.cuda.empty_cache()

initial target device:   0%|          | 0/2 [00:00<?, ?it/s]WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
E0000 00:00:1774787013.283365      90 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774787013.291056      90 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774787013.310632      90 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774787013.310686      90 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774787013.310692      90 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the sam